# Sistema de Recomendação por Similaridade de Itens

Este notebook implementa um sistema de recomendação colaborativo baseado em **similaridade entre itens** (Item-Based Collaborative Filtering).

## Como o Algoritmo Funciona

A ideia central é: *filmes que foram avaliados de forma parecida pelos mesmos usuários são similares entre si*. O algoritmo opera em três etapas:

### 1. Cálculo de Similaridade entre Itens (Pearson)
Para cada par de filmes $i$ e $j$, calculamos a **Correlação de Pearson** sobre os usuários que avaliaram ambos:

$$\text{sim}(i, j) = \frac{\sum_{u \in U_{ij}} (r_{u,i} - \bar{r}_i)(r_{u,j} - \bar{r}_j)}{\sqrt{\sum_{u \in U_{ij}} (r_{u,i} - \bar{r}_i)^2} \cdot \sqrt{\sum_{u \in U_{ij}} (r_{u,j} - \bar{r}_j)^2}}$$

Onde $U_{ij}$ é o conjunto de usuários que avaliaram *ambos* os filmes $i$ e $j$, e $\bar{r}_i$ é a nota média do filme $i$ em todo o catálogo.

### 2. Predição de Score (ajustada pelo viés do item)
Para recomendar o filme $i$ a um usuário $u$ que ainda não o viu, buscamos os **K filmes mais similares a $i$** que o usuário já avaliou:

$$\hat{r}_{u,i} = \bar{r}_i + \frac{\sum_{j \in N_K(i) \cap I_u} \text{sim}(i,j) \cdot (r_{u,j} - \bar{r}_j)}{\sum_{j \in N_K(i) \cap I_u} |\text{sim}(i,j)|}$$

Onde $N_K(i)$ são os K filmes mais similares a $i$, e $I_u$ são os filmes que o usuário $u$ já assistiu.

O ajuste $(r_{u,j} - \bar{r}_j)$ remove o viés do item: filmes populares tendem a ter notas altas de todos, então usamos o *desvio* em relação à média do filme.

### 3. Geração do Ranking
Com os scores calculados, geramos duas variantes de ranking:
- **Top-N Determinístico**: ordena por score e retorna os N melhores.
- **Top-N Estocástico**: converte scores em probabilidades (softmax), recomendando com aleatoriedade controlada.

## Item-Based vs User-Based

| Aspecto | User-Based | Item-Based |
|---------|------------|------------|
| Similaridade | Entre usuários | Entre filmes |
| Estabilidade | Baixa (gostos mudam) | Alta (filmes não mudam) |
| Escalabilidade | Ruim para muitos usuários | Ruim para muitos filmes |
| Precomputação | Rápida se n_usuários pequeno | Pode ser cacheada em disco |
| Explicabilidade | "Usuários como você gostaram" | "Por ter gostado de X, recomendo Y" |

## Dados Utilizados

| Arquivo | Conteúdo |
|---------|----------|
| `movie_encoding.tsv` | Features dos filmes (gêneros, idiomas, países) |
| `user_ratings.csv` | Avaliações dos usuários (USERID, MOVIEID, RATING) |
| `recommendations.tsv` | Histórico de recomendações anteriores (opcional) |
| `movie_category.tsv` | Categorias de diversidade para métricas de Commonality |

In [ ]:
!pip install numpy pandas scipy matplotlib tqdm --quiet

import numpy, pandas, scipy, matplotlib
print(f'numpy {numpy.__version__} | pandas {pandas.__version__} | scipy {scipy.__version__}')

In [ ]:
import os
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

print('Bibliotecas carregadas com sucesso.')

## 1. Upload dos Arquivos

Monte o Google Drive e configure os caminhos para os arquivos abaixo.

| Arquivo | Descrição | Obrigatório |
|---------|-----------|-------------|
| `movie_encoding.tsv` | Features dos filmes — gêneros, idiomas, países (gerado por `rs_movie_encoding.py`) | Sim |
| `user_ratings.csv` | Avaliações dos usuários: colunas `USERID`, `MOVIEID`, `RATING` | Sim |
| `recommendations.tsv` | Histórico de recomendações passadas: colunas `USERID`, `MOVIEID` | Não |
| `movie_category.tsv` | Categorias de diversidade: `director_women`, `director_nowhite`, etc. (gerado por `rs_movie_category.py`) | Para Commonality |

> **Dica**: coloque todos os arquivos em uma pasta no Google Drive (ex: `MinhaUnidade/rs_data/`) e ajuste `DRIVE_BASE` abaixo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Configure os caminhos aqui ────────────────────────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/rs_data/'

MOVIE_ENC_PATH = os.path.join(DRIVE_BASE, 'movie_encoding.tsv')
RATINGS_PATH   = os.path.join(DRIVE_BASE, 'user_ratings.csv')
RECS_PATH      = os.path.join(DRIVE_BASE, 'recommendations.tsv')   # opcional
CATEGORY_PATH  = os.path.join(DRIVE_BASE, 'movie_category.tsv')    # para Commonality

# Verificação
for label, path, required in [
    ('movie_encoding.tsv', MOVIE_ENC_PATH, True),
    ('user_ratings.csv',   RATINGS_PATH,   True),
    ('recommendations.tsv',RECS_PATH,      False),
    ('movie_category.tsv', CATEGORY_PATH,  False),
]:
    exists = os.path.exists(path)
    status = '✓' if exists else ('✗ OBRIGATÓRIO' if required else '— opcional')
    print(f'  {status}  {label}')

## 2. Carregamento e Exploração dos Dados

Antes de treinar o modelo, é importante entender a distribuição dos dados:

- **Esparsidade**: em sistemas de recomendação, a maioria dos usuários avalia apenas uma fração pequena do catálogo. No Item-Based CF, a esparsidade afeta diretamente o número de usuários em comum que dois filmes têm — pares de filmes com poucos avaliadores em comum terão similaridades menos confiáveis.
- **Cauda longa**: filmes com poucas avaliações terão similaridades instáveis. O parâmetro `MIN_COMMON_USERS` filtrará esses pares.
- **Tamanho do catálogo**: com muitos filmes, a matriz de similaridade item-item ($n_{filmes} \times n_{filmes}$) pode ser grande — o cálculo é feito em **batches** para controlar a memória.

Os gráficos abaixo permitem identificar esses padrões nos seus dados.

In [ ]:
# ── Carrega os dados ──────────────────────────────────────────────────────────
print('Carregando movie_encoding.tsv ...')
movies_df = pd.read_csv(MOVIE_ENC_PATH, sep='\t', low_memory=False)
genre_cols = [c for c in movies_df.columns if c.startswith('genre_')]
print(f'  {len(movies_df):,} filmes | {len(genre_cols)} gêneros')

print('Carregando user_ratings.csv ...')
ratings_df = pd.read_csv(RATINGS_PATH)
ratings_df.columns = ratings_df.columns.str.upper()
ratings_df = ratings_df.dropna(subset=['USERID','MOVIEID','RATING'])
ratings_df['USERID']  = ratings_df['USERID'].astype(int)
ratings_df['MOVIEID'] = ratings_df['MOVIEID'].astype(int)
ratings_df['RATING']  = ratings_df['RATING'].astype(float)
print(f'  {len(ratings_df):,} avaliações | {ratings_df["USERID"].nunique():,} usuários | {ratings_df["MOVIEID"].nunique():,} filmes')

if os.path.exists(RECS_PATH):
    recs_df = pd.read_csv(RECS_PATH, sep='\t')
    recs_df.columns = recs_df.columns.str.upper()
    print(f'Recomendações anteriores: {len(recs_df):,}')
else:
    recs_df = None
    print('recommendations.tsv não encontrado — histórico de sistema vazio.')

# ── Gráficos exploratórios ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploração dos Dados de Avaliações', fontsize=14, fontweight='bold')

# 1. Distribuição de ratings
ax = axes[0, 0]
ratings_df['RATING'].hist(bins=20, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribuição de Ratings')
ax.set_xlabel('Rating')
ax.set_ylabel('Frequência')
ax.axvline(ratings_df['RATING'].mean(), color='crimson', linestyle='--',
           label=f'Média: {ratings_df["RATING"].mean():.2f}')
ax.legend()

# 2. Avaliações por usuário
ax = axes[0, 1]
n_per_user = ratings_df.groupby('USERID').size()
ax.hist(n_per_user, bins=50, color='darkorange', edgecolor='white', log=True)
ax.set_title('Avaliações por Usuário (escala log)')
ax.set_xlabel('Número de filmes avaliados')
ax.set_ylabel('Nº de usuários (log)')
ax.axvline(n_per_user.median(), color='crimson', linestyle='--',
           label=f'Mediana: {n_per_user.median():.0f}')
ax.legend()

# 3. Avaliações por filme (crucial para item-based)
ax = axes[1, 0]
n_per_movie = ratings_df.groupby('MOVIEID').size()
ax.hist(n_per_movie, bins=50, color='seagreen', edgecolor='white', log=True)
ax.set_title('Avaliações por Filme (escala log)\n⚠ Filmes com poucas avaliações terão similaridade instável')
ax.set_xlabel('Número de avaliações')
ax.set_ylabel('Nº de filmes (log)')
ax.axvline(n_per_movie.median(), color='crimson', linestyle='--',
           label=f'Mediana: {n_per_movie.median():.0f}')
ax.legend()

# 4. Top gêneros
ax = axes[1, 1]
if genre_cols:
    genre_counts = movies_df[genre_cols].sum().sort_values(ascending=False).head(12)
    genre_counts.index = genre_counts.index.str.replace('genre_', '').str.replace('_', ' ').str.title()
    genre_counts.plot(kind='barh', ax=ax, color='mediumpurple')
    ax.set_title('Top-12 Gêneros no Catálogo')
    ax.set_xlabel('Número de filmes')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

n_users_total  = ratings_df['USERID'].nunique()
n_movies_total = ratings_df['MOVIEID'].nunique()
sparsity = 1 - len(ratings_df) / (n_users_total * n_movies_total)
print(f'\nEsparsidade da matriz: {sparsity:.4%} ({n_users_total} usuários × {n_movies_total} filmes)')
print(f'Matriz de similaridade item-item seria: {n_movies_total:,} × {n_movies_total:,} = {n_movies_total**2:,} entradas')
print(f'(armazenamos apenas top-K por filme — ver parâmetro TOP_K_SIMILAR)')

## 3. Divisão Treino / Teste por Usuário (80/20)

Para avaliar o algoritmo, dividimos as avaliações de **cada usuário individualmente** em 80% treino e 20% teste.

A divisão **por usuário** é essencial porque:
- Garante que todo usuário tenha dados no treino para servir como referência na predição de scores.
- Garante que todo usuário tenha dados no teste para ser avaliado.
- Evita que um filme apareça apenas no teste (sem avaliações no treino, não há similaridade calculada para ele).

**Parâmetros configuráveis:**
- `TEST_RATIO`: proporção do teste (padrão: 0.2 = 20%).
- `MIN_RATINGS_SPLIT`: usuários com menos que este número de ratings ficam inteiramente no treino.
- `RELEVANCE_THRESHOLD`: rating mínimo para considerar um filme como "relevante" nas métricas.

In [ ]:
# ── Parâmetros configuráveis ──────────────────────────────────────────────────
TEST_RATIO          = 0.20   # 20% dos ratings de cada usuário vão para teste
MIN_RATINGS_SPLIT   = 5      # usuários com menos que isso vão 100% para treino
RELEVANCE_THRESHOLD = 4.0    # rating >= este valor é considerado "relevante"

# ── Split por usuário ─────────────────────────────────────────────────────────
train_ratings: dict[int, dict[int, float]] = defaultdict(dict)  # {uid: {mid: rating}}
test_ratings:  dict[int, dict[int, float]] = defaultdict(dict)

for uid, group in ratings_df.groupby('USERID'):
    rows = group[['MOVIEID', 'RATING']].values.tolist()
    random.shuffle(rows)

    if len(rows) < MIN_RATINGS_SPLIT:
        for mid, rat in rows:
            train_ratings[uid][int(mid)] = float(rat)
    else:
        n_test = max(1, int(len(rows) * TEST_RATIO))
        for mid, rat in rows[n_test:]:
            train_ratings[uid][int(mid)] = float(rat)
        for mid, rat in rows[:n_test]:
            test_ratings[uid][int(mid)]  = float(rat)

train_ratings = dict(train_ratings)
test_ratings  = dict(test_ratings)

train_users      = [u for u in train_ratings if len(train_ratings[u]) >= MIN_RATINGS_SPLIT]
all_movies_train = sorted(set(m for u in train_ratings.values() for m in u))

n_test_users = sum(1 for u in train_users if u in test_ratings and
                   any(r >= RELEVANCE_THRESHOLD for r in test_ratings[u].values()))

print(f'Usuários totais      : {ratings_df["USERID"].nunique():,}')
print(f'Usuários no treino   : {len(train_users):,}')
print(f'Usuários com teste   : {len(test_ratings):,}')
print(f'  (com item relevante): {n_test_users:,}')
print(f'Filmes no catálogo   : {len(all_movies_train):,}')
print(f'\nRatings no treino    : {sum(len(v) for v in train_ratings.values()):,}')
print(f'Ratings no teste     : {sum(len(v) for v in test_ratings.values()):,}')

## 4. Similaridade de Pearson entre Filmes

A **Correlação de Pearson** entre dois filmes $i$ e $j$ mede o quanto as notas que os usuários deram a ambos concordam, após remover o viés de cada filme (sua nota média).

### Por que normalizar pela média do item?

Filmes populares e aclamados (ex: *O Poderoso Chefão*) recebem notas altas de quase todos — o que tornaria qualquer par com esse filme "similar" por razões erradas. Subtraindo a média do item, focamos no *padrão relativo* de avaliação:

| Usuário | Filme A | Filme B |
|---------|---------|--------|
| Maria | 5 | 4 |
| João | 3 | 2 |
| Ana | 4 | 3 |

Médias: A=4, B=3. Desvios de A: [+1, -1, 0]. Desvios de B: [+1, -1, 0]. Pearson = 1.0 — gostos idênticos, escala diferente.

### Implementação Eficiente

1. Constrói a matriz **filme × usuário** (transposta da user-item).
2. Normaliza cada linha (filme) subtraindo sua média de rating.
3. Calcula o produto matricial `R_norm @ R_norm.T` → matriz de similaridade item-item.
4. Para cada filme, mantém apenas os **TOP_K_SIMILAR** mais similares.

> ⚠️ Com catálogos grandes ($n_{filmes} \gg n_{usuarios}$), esta matriz pode ser pesada. O cálculo é feito em **batches** para controlar a memória.

In [ ]:
# ── Parâmetros configuráveis ──────────────────────────────────────────────────
TOP_K_SIMILAR    = 50    # vizinhos mais similares armazenados por filme
MIN_COMMON_USERS = 3     # mínimo de usuários em comum para calcular similaridade
SIM_BATCH_SIZE   = 500   # tamanho do batch para cálculo matricial

# ── Constrói a matriz filme × usuário ─────────────────────────────────────────
# (transposta da user-item usada no User-Based)
all_user_ids = sorted(set(u for u in train_ratings))
movie_to_idx = {m: i for i, m in enumerate(all_movies_train)}
user_to_idx  = {u: i for i, u in enumerate(all_user_ids)}

n_movies_train = len(all_movies_train)
n_users_train  = len(all_user_ids)

print(f'Construindo matriz {n_movies_train:,} filmes × {n_users_train:,} usuários ...')

# R[filme_idx, user_idx] = rating (NaN se não avaliado)
R = np.full((n_movies_train, n_users_train), np.nan, dtype=np.float32)

for uid, movies in train_ratings.items():
    j = user_to_idx.get(uid)
    if j is None:
        continue
    for mid, rat in movies.items():
        i = movie_to_idx.get(mid)
        if i is not None:
            R[i, j] = rat

# ── Normaliza por filme (subtrai média do item) → Pearson = cosseno normalizado
item_means_arr = np.nanmean(R, axis=1, keepdims=True)   # (n_movies, 1)
R_norm = np.where(np.isnan(R), 0.0, R - item_means_arr)  # NaN → 0 após subtração

# Matriz binária para contar usuários em comum
R_bin = (~np.isnan(R)).astype(np.float32)  # 1 se avaliado

norms = np.linalg.norm(R_norm, axis=1, keepdims=True)
norms = np.where(norms < 1e-10, 1e-10, norms)
R_unit = R_norm / norms  # (n_movies, n_users) — vetores unitários

# ── Calcula similaridade em batches ──────────────────────────────────────────
item_similarity: dict[int, list[tuple[int, float]]] = {}  # {mid: [(other_mid, sim), ...]}

print(f'Calculando similaridade de Pearson entre filmes em batches de {SIM_BATCH_SIZE}...')
for i_start in tqdm(range(0, n_movies_train, SIM_BATCH_SIZE), desc='Batches'):
    i_end = min(i_start + SIM_BATCH_SIZE, n_movies_train)

    sim_block    = R_unit[i_start:i_end] @ R_unit.T      # (batch, n_movies)
    common_block = R_bin[i_start:i_end]  @ R_bin.T       # (batch, n_movies)

    for bi, i in enumerate(range(i_start, i_end)):
        row = sim_block[bi].copy()
        com = common_block[bi]

        row[i]   = -2.0                       # exclui si mesmo
        row[com < MIN_COMMON_USERS] = -2.0    # filtra pares com poucos usuários em comum

        top_idx = np.argsort(row)[::-1][:TOP_K_SIMILAR]
        mid = all_movies_train[i]
        item_similarity[mid] = [
            (all_movies_train[j], float(row[j]))
            for j in top_idx if row[j] > -1.5
        ]

del R_norm, R_unit, R_bin

# Resumo
all_sims = [s for sims in item_similarity.values() for _, s in sims]
print(f'\nSimilaridade calculada!')
print(f'  Filmes com ao menos 1 vizinho: {sum(1 for v in item_similarity.values() if v):,}')
print(f'  Similaridade média            : {np.mean(all_sims):.4f}')
print(f'  Vizinhos médios por filme     : {np.mean([len(v) for v in item_similarity.values()]):.1f}')

## 5. Cálculo de Score por Filme

Para cada usuário $u$, queremos predizer a nota que ele daria a cada filme $i$ que ainda não assistiu.

A abordagem é: para cada filme $i$ candidato, olhamos os **K filmes mais similares a $i$** que o usuário $u$ **já avaliou** e fazemos uma média ponderada pela similaridade:

$$\hat{r}_{u,i} = \bar{r}_i + \frac{\sum_{j \in N_K(i) \cap I_u} \text{sim}(i,j) \cdot (r_{u,j} - \bar{r}_j)}{\sum_{j \in N_K(i) \cap I_u} |\text{sim}(i,j)|}$$

### Estratégia de Acumulação (Eficiente)

Em vez de iterar sobre cada candidato $i$ e buscar seus vizinhos, invertemos o processo:

1. Para cada filme $j$ que o usuário **já avaliou** (conjunto pequeno):
2. Para cada filme $i$ similar a $j$ (top-K vizinhos de $j$):
3. Se $i$ não foi avaliado por $u$: acumula `sim(i,j) * (r_u,j - r̄_j)` no score de $i$.

Complexidade: $O(|I_u| \times K)$ por usuário — muito mais eficiente do que $O(n_{filmes} \times K)$.

### Parâmetros

- `K_NEIGHBORS`: quantos vizinhos de cada filme avaliado usar (recomendado: 10–50).
- `MIN_NEIGHBORS`: mínimo de filmes avaliados que contribuíram para o score (evita predições com base em 1 filme).
- `GLOBAL_MEAN_FALLBACK`: média global usada como fallback para filmes sem média calculada.

In [ ]:
# ── Parâmetros configuráveis ──────────────────────────────────────────────────
K_NEIGHBORS   = 20   # vizinhos de cada filme avaliado usados na predição
MIN_NEIGHBORS = 2    # mínimo de contribuições para gerar um score

# ── Médias por item (precomputadas) ──────────────────────────────────────────
global_mean = float(ratings_df['RATING'].mean())

item_mean_rating: dict[int, float] = {}
for i, mid in enumerate(all_movies_train):
    mean_val = float(item_means_arr[i, 0])
    item_mean_rating[mid] = mean_val if not np.isnan(mean_val) else global_mean

# ── Cálculo de scores via acumulação inversa ──────────────────────────────────
# Para cada usuário: itera sobre filmes já avaliados e propaga para filmes similares
user_scores: dict[int, dict[int, float]] = {}

for uid in tqdm(train_users, desc='Calculando scores'):
    rated_by_u = train_ratings[uid]       # {mid: rating}

    score_num = defaultdict(float)         # {candidate_mid: numerador acumulado}
    score_den = defaultdict(float)         # {candidate_mid: denominador acumulado}
    score_cnt = defaultdict(int)           # {candidate_mid: contribuições contadas}

    for j_mid, r_uj in rated_by_u.items():
        j_mean = item_mean_rating.get(j_mid, global_mean)
        dev_j  = r_uj - j_mean            # desvio do usuário em relação à média do filme j

        for i_mid, sim in item_similarity.get(j_mid, [])[:K_NEIGHBORS]:
            if i_mid not in rated_by_u:   # só filmes não assistidos
                score_num[i_mid] += sim * dev_j
                score_den[i_mid] += abs(sim)
                score_cnt[i_mid] += 1

    scores = {}
    for i_mid in score_num:
        if score_den[i_mid] > 0 and score_cnt[i_mid] >= MIN_NEIGHBORS:
            i_mean = item_mean_rating.get(i_mid, global_mean)
            scores[i_mid] = i_mean + score_num[i_mid] / score_den[i_mid]

    user_scores[uid] = scores

# Resumo
n_scored          = sum(len(s) for s in user_scores.values())
n_users_with_scores = sum(1 for s in user_scores.values() if s)
print(f'\nScores calculados!')
print(f'  Usuários com ao menos 1 score : {n_users_with_scores:,}')
print(f'  Total de (usuário, filme) com score: {n_scored:,}')
if n_scored > 0:
    all_score_vals = [s for sc in user_scores.values() for s in sc.values()]
    print(f'  Score médio : {np.mean(all_score_vals):.3f}')
    print(f'  Score mín   : {np.min(all_score_vals):.3f}')
    print(f'  Score máx   : {np.max(all_score_vals):.3f}')

## 6. Geração do Ranking de Recomendações

Com os scores calculados, geramos duas estratégias de ranking:

### Top-N Determinístico
Ordena todos os filmes com score pelo valor decrescente e retorna os **N primeiros**. É determinístico: para os mesmos dados, sempre produz o mesmo resultado.

**Vantagem**: Maximiza a relevância esperada.  
**Desvantagem**: Baixa diversidade — tende a sempre recomendar os mesmos filmes populares.

### Top-N Estocástico
Converte os scores em **probabilidades** via softmax e amostra N filmes **sem reposição**. Filmes com score abaixo de `MIN_SCORE` são excluídos.

$$p(i) = \frac{e^{\hat{r}_{u,i} / T}}{\sum_{j} e^{\hat{r}_{u,j} / T}}$$

O parâmetro de **temperatura** $T$ controla o grau de aleatoriedade:
- $T \to 0$: concentra toda a probabilidade no item de maior score (equivale a Top-1).
- $T = 1$: distribuição proporcional ao score.
- $T \to \infty$: distribuição uniforme (completamente aleatório).

**Vantagem**: Maior diversidade e cobertura do catálogo.  
**Desvantagem**: Pode incluir itens de menor relevância.

### Parâmetros
- `RANKING_METHOD`: `'topn'` ou `'stochastic'`
- `N_RECOMMENDATIONS`: número de filmes por recomendação
- `MIN_SCORE_STOCHASTIC`: score mínimo para entrar no sorteio estocástico
- `TEMPERATURE`: temperatura do softmax (somente para estocástico)

In [ ]:
# ── Parâmetros configuráveis ──────────────────────────────────────────────────
RANKING_METHOD       = 'topn'   # 'topn' | 'stochastic'
N_RECOMMENDATIONS    = 10       # filmes por usuário
MIN_SCORE_STOCHASTIC = 3.0      # score mínimo para ranking estocástico
TEMPERATURE          = 1.0      # temperatura do softmax

# ── Funções de ranking ────────────────────────────────────────────────────────

def top_n_ranking(scores: dict, n: int, exclude: set) -> list:
    filtered = {m: s for m, s in scores.items() if m not in exclude}
    return sorted(filtered, key=filtered.get, reverse=True)[:n]


def stochastic_ranking(scores: dict, n: int, min_score: float,
                        temperature: float, exclude: set) -> list:
    eligible = {m: s for m, s in scores.items()
                if s >= min_score and m not in exclude}
    if not eligible:
        return top_n_ranking(scores, n, exclude)

    movies = list(eligible.keys())
    raw_s  = np.array([eligible[m] for m in movies], dtype=np.float64)
    logits = raw_s / max(temperature, 1e-8)
    exp_s  = np.exp(logits - logits.max())
    probs  = exp_s / exp_s.sum()

    k = min(n, len(movies))
    chosen = np.random.choice(len(movies), size=k, replace=False, p=probs)
    return [movies[i] for i in chosen]


# ── Gera rankings para todos os usuários ─────────────────────────────────────
user_rankings: dict[int, list] = {}

for uid in tqdm(train_users, desc='Gerando rankings'):
    scores  = user_scores.get(uid, {})
    exclude = set(train_ratings[uid].keys())

    if RANKING_METHOD == 'stochastic':
        ranking = stochastic_ranking(scores, N_RECOMMENDATIONS,
                                      MIN_SCORE_STOCHASTIC, TEMPERATURE, exclude)
    else:
        ranking = top_n_ranking(scores, N_RECOMMENDATIONS, exclude)

    user_rankings[uid] = ranking

lens = [len(r) for r in user_rankings.values()]
print(f'Rankings gerados: {len(user_rankings):,} usuários')
print(f'  Método        : {RANKING_METHOD.upper()}')
print(f'  Tamanho médio : {np.mean(lens):.1f} filmes')
print(f'  Com ranking   : {sum(1 for l in lens if l > 0):,} usuários')

## 7. Métricas de Avaliação

Avaliamos a qualidade do ranking com as seguintes métricas, implementadas em `rs_metrics.py`:

| Métrica | O que mede | Fórmula resumida |
|---------|------------|------------------|
| **Precision@K** | De K recomendações, quantas são relevantes? | `|top-K ∩ relevantes| / K` |
| **Recall@K** | De todos os relevantes, quantos estão no top-K? | `|top-K ∩ relevantes| / |relevantes|` |
| **F1@K** | Equilíbrio entre Precision e Recall | `2·P·R / (P+R)` |
| **NDCG@K** | Qualidade do ranking ponderando pela posição | Ganho descontado normalizado |
| **MAP** | Precisão média ponderada pela posição do primeiro acerto | Média dos Average Precisions |
| **Hit Rate** | % de usuários com ao menos 1 item relevante | `usuários_com_hit / total` |
| **MRR** | Posição média do primeiro item relevante | `média(1/posição_do_primeiro_hit)` |
| **Coverage** | % do catálogo que o sistema recomenda | `|filmes_únicos_recomendados| / |catálogo|` |

**Itens relevantes**: filmes no conjunto de teste com `RATING >= RELEVANCE_THRESHOLD` (configurado na célula de split).

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/rs_data/')   # pasta com rs_metrics.py
from rs_metrics import RankingMetrics

# ── Parâmetros configuráveis ──────────────────────────────────────────────────
K_METRICS = N_RECOMMENDATIONS

# ── Prepara listas de (ranking, relevantes) ───────────────────────────────────
rankings_eval = []
relevant_eval = []
eval_user_ids = []

for uid in train_users:
    if uid not in user_rankings or not user_rankings[uid]:
        continue
    if uid not in test_ratings:
        continue

    relevant = [m for m, r in test_ratings[uid].items() if r >= RELEVANCE_THRESHOLD]
    if not relevant:
        continue

    rankings_eval.append(user_rankings[uid])
    relevant_eval.append(relevant)
    eval_user_ids.append(uid)

print(f'Usuários avaliados: {len(eval_user_ids):,}')

# ── Calcula métricas ──────────────────────────────────────────────────────────
m = RankingMetrics()

prec   = np.mean([m.precision_at_k(r, rel, K_METRICS) for r, rel in zip(rankings_eval, relevant_eval)])
rec    = np.mean([m.recall_at_k(r, rel, K_METRICS)    for r, rel in zip(rankings_eval, relevant_eval)])
ndcg   = np.mean([m.ndcg_at_k(r, rel, K_METRICS)      for r, rel in zip(rankings_eval, relevant_eval)])
f1     = m.f1_score(prec, rec)
map_s  = m.mean_average_precision(rankings_eval, relevant_eval)
hr     = m.hit_rate(rankings_eval, relevant_eval)
mrr    = m.mean_reciprocal_rank(rankings_eval, relevant_eval)
cov    = m.coverage(rankings_eval, catalog_size=len(all_movies_train))

metric_results = {
    f'Precision@{K_METRICS}': prec,
    f'Recall@{K_METRICS}':    rec,
    f'F1@{K_METRICS}':        f1,
    f'NDCG@{K_METRICS}':      ndcg,
    'MAP':                     map_s,
    'Hit Rate':                hr,
    'MRR':                     mrr,
    'Coverage':                cov,
}

print(f'\n{"Métrica":<20}  {"Valor":>8}')
print('─' * 32)
for name, val in metric_results.items():
    print(f'{name:<20}  {val:>8.4f}')

## 8. Métrica de Commonality (Diversidade de Exposição)

A métrica **Commonality**, implementada em `rs_commonality.py`, mede a probabilidade de que **todos os usuários simultaneamente** se familiarizem com uma categoria de filmes.

### Conceitos

**Probabilidade de Browsing** (baseada no modelo RBP — Rank-Biased Precision):
$$\Pr(k) = (1 - \gamma) \cdot \gamma^{k-1}$$
A probabilidade de um usuário ainda estar vendo a posição $k$. Com $\gamma = 0.8$: posição 1 tem 20% de chance, posição 5 tem ~8%, posição 10 tem ~3%.

**Familiaridade** de um usuário $u$ com uma categoria $g$ dado seu ranking $\pi_u$:
$$\Pr(F_{u,g} | \pi_u) = \sum_{k=1}^{N} \Pr(k) \cdot R(\pi_u, k, g)$$
onde $R(\pi_u, k, g)$ é o Recall da categoria nos top-$k$ itens.

**Commonality** de uma categoria para todos os usuários:
$$C_g(\pi) = \prod_{u} \Pr(F_{u,g} | \pi_u)$$
O produto penaliza fortemente casos em que qualquer usuário tem familiaridade próxima de zero.

### Categorias de Diversidade

As categorias definidas para cotas de tela são extraídas de `movie_category.tsv`:

| Categoria | Critério |
|-----------|----------|
| Diretoras Mulheres | `director_women = 1` |
| Dir. Raça Não-Branca | `director_nowhite = 1` |
| Dir. Região Não-EU/NA | `director_region = 1` |
| Origem Não-EU/NA | `movie_region = 1` |
| Produção Brasileira | `movie_region_bra = 1` |

> **Nota**: A Commonality é calculada sobre uma **amostra** de usuários para viabilidade computacional.

In [ ]:
from rs_commonality import CommonalityCalculator

# ── Parâmetros configuráveis ──────────────────────────────────────────────────
GAMMA              = 0.8    # paciência do usuário ao navegar o ranking
COMMONALITY_SAMPLE = 200    # usuários amostrados para o cálculo (produto)

# ── Carrega categorias ────────────────────────────────────────────────────────
if not os.path.exists(CATEGORY_PATH):
    raise FileNotFoundError(
        f'Arquivo não encontrado: {CATEGORY_PATH}\n'
        'Execute rs_movie_category.py para gerar movie_category.tsv.'
    )

cat_df = pd.read_csv(CATEGORY_PATH, sep='\t')
print(f'Categorias carregadas: {len(cat_df):,} filmes')

DIVERSITY_CATEGORIES = {
    'Diretoras Mulheres':     set(cat_df[cat_df['director_women']   == 1]['movieid']),
    'Dir. Raça Não-Branca':   set(cat_df[cat_df['director_nowhite'] == 1]['movieid']),
    'Dir. Reg. Não-EU/NA':    set(cat_df[cat_df['director_region']  == 1]['movieid']),
    'Origem Não-EU/NA':       set(cat_df[cat_df['movie_region']     == 1]['movieid']),
    'Produção Brasileira':    set(cat_df[cat_df['movie_region_bra'] == 1]['movieid']),
}

for cat, items in DIVERSITY_CATEGORIES.items():
    print(f'  {cat:<26}: {len(items):,} filmes')

# ── Amostra de rankings para o cálculo ───────────────────────────────────────
eligible_users  = [u for u in train_users if user_rankings.get(u)]
sample_size     = min(COMMONALITY_SAMPLE, len(eligible_users))
sample_uids     = random.sample(eligible_users, sample_size)
sample_rankings = [user_rankings[u] for u in sample_uids]

print(f'\nAmostra de {sample_size} usuários para Commonality (γ={GAMMA})')

# ── Calcula Familiaridade e Commonality por categoria ────────────────────────
calc = CommonalityCalculator(gamma=GAMMA)

commonality_results = {}
print(f'\n{"Categoria":<26}  {"Fam. Média":>10}  {"Fam. Std":>9}  {"Commonality":>12}')
print('─' * 65)

for cat_name, cat_items in DIVERSITY_CATEGORIES.items():
    cat_list = list(cat_items)

    familiarities = [
        calc.familiarity(ranking, cat_list)
        for ranking in sample_rankings
    ]

    commonality = calc.categoria_commonality(sample_rankings, cat_list)

    commonality_results[cat_name] = {
        'fam_mean':    float(np.mean(familiarities)),
        'fam_std':     float(np.std(familiarities)),
        'commonality': float(commonality),
        'n_items':     len(cat_items),
    }

    fam_m = commonality_results[cat_name]['fam_mean']
    fam_s = commonality_results[cat_name]['fam_std']
    comm  = commonality_results[cat_name]['commonality']
    print(f'{cat_name:<26}  {fam_m:>10.4f}  {fam_s:>9.4f}  {comm:>12.6f}')

print(f'\nNota: Commonality é o produto das familiaridades — decresce com mais usuários.')

## 9. Visualização dos Resultados

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    f'Resultados — Item-Based CF  |  Método: {RANKING_METHOD.upper()}  |  '
    f'Top-{N_RECOMMENDATIONS}  |  K={K_NEIGHBORS} vizinhos por item',
    fontsize=13, fontweight='bold'
)

# ── 1. Métricas de avaliação ──────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
metric_names = list(metric_results.keys())
metric_vals  = list(metric_results.values())
colors_m = ['steelblue' if v > 0.1 else 'lightsteelblue' for v in metric_vals]
bars = ax1.barh(metric_names, metric_vals, color=colors_m, edgecolor='white')
ax1.set_xlim(0, max(metric_vals) * 1.25)
ax1.set_title('Métricas de Avaliação', fontweight='bold')
ax1.set_xlabel('Valor')
for bar, val in zip(bars, metric_vals):
    ax1.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
             f'{val:.4f}', va='center', fontsize=8)
ax1.invert_yaxis()

# ── 2. Familiaridade média por categoria ─────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2)
cat_names = list(commonality_results.keys())
fam_means = [commonality_results[c]['fam_mean'] for c in cat_names]
fam_stds  = [commonality_results[c]['fam_std']  for c in cat_names]
y_pos = range(len(cat_names))
ax2.barh(list(y_pos), fam_means, xerr=fam_stds, color='darkorange',
         edgecolor='white', capsize=4, alpha=0.8)
ax2.set_yticks(list(y_pos))
ax2.set_yticklabels(cat_names, fontsize=9)
ax2.set_title('Familiaridade Média por Categoria\n(com desvio padrão)', fontweight='bold')
ax2.set_xlabel('Familiaridade esperada')
ax2.invert_yaxis()

# ── 3. Commonality por categoria ──────────────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
comm_vals = [commonality_results[c]['commonality'] for c in cat_names]
ax3.barh(list(y_pos), comm_vals, color='seagreen', edgecolor='white', alpha=0.8)
ax3.set_yticks(list(y_pos))
ax3.set_yticklabels(cat_names, fontsize=9)
ax3.set_title(f'Commonality por Categoria\n(amostra {sample_size} usuários)', fontweight='bold')
ax3.set_xlabel('Commonality (produto das familiaridades)')
ax3.xaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax3.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
ax3.invert_yaxis()

# ── 4. Distribuição de scores ─────────────────────────────────────────────────
ax4 = fig.add_subplot(2, 3, 4)
flat_scores = [s for sc in user_scores.values() for s in sc.values()]
if flat_scores:
    ax4.hist(flat_scores, bins=50, color='mediumpurple', edgecolor='white', alpha=0.8)
    ax4.axvline(np.mean(flat_scores), color='crimson', linestyle='--',
                label=f'Média: {np.mean(flat_scores):.2f}')
    ax4.axvline(global_mean, color='orange', linestyle=':',
                label=f'Média global: {global_mean:.2f}')
    ax4.set_title('Distribuição de Scores Preditos', fontweight='bold')
    ax4.set_xlabel('Score predito')
    ax4.set_ylabel('Frequência')
    ax4.legend(fontsize=8)

# ── 5. Tamanho das categorias de diversidade ──────────────────────────────────
ax5 = fig.add_subplot(2, 3, 5)
n_items_cat = [commonality_results[c]['n_items'] for c in cat_names]
total_catalog = len(all_movies_train)
pcts = [v / total_catalog * 100 if total_catalog > 0 else 0 for v in n_items_cat]
ax5.barh(list(y_pos), pcts, color='steelblue', edgecolor='white', alpha=0.8)
ax5.set_yticks(list(y_pos))
ax5.set_yticklabels(cat_names, fontsize=9)
ax5.set_title(f'Tamanho das Categorias\n(% do catálogo — {total_catalog:,} filmes)', fontweight='bold')
ax5.set_xlabel('% do catálogo')
for i, (pct, n) in enumerate(zip(pcts, n_items_cat)):
    ax5.text(pct + 0.3, i, f'{n:,} ({pct:.1f}%)', va='center', fontsize=8)
ax5.invert_yaxis()

# ── 6. Distribuição de similaridade item-item ─────────────────────────────────
ax6 = fig.add_subplot(2, 3, 6)
sim_sample = [s for sims in item_similarity.values() for _, s in sims[:10]]
if sim_sample:
    ax6.hist(sim_sample, bins=40, color='teal', edgecolor='white', alpha=0.8)
    ax6.axvline(np.mean(sim_sample), color='crimson', linestyle='--',
                label=f'Média: {np.mean(sim_sample):.3f}')
    ax6.set_title('Distribuição de Similaridade Item-Item\n(top-10 vizinhos por filme)', fontweight='bold')
    ax6.set_xlabel('Correlação de Pearson')
    ax6.set_ylabel('Frequência')
    ax6.legend(fontsize=8)

plt.tight_layout()
plt.savefig('resultados_item_sim.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em resultados_item_sim.png')